In [1]:
from pathlib import Path
import sys

In [2]:
project_root = Path.cwd().resolve().parent
project_root

PosixPath('/bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10')

In [3]:
sys.path.insert(0, str(project_root))

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from torchvision import datasets, transforms

from models.vit import VisionTransformer

In [5]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [ ]:
IMAGE_SIZE = 32
BATCH_SIZE = 256
PATCH_SIZE = 8
IN_CHANNELS = 3
EMB_SIZE = 384
NUM_ATTENTION_HEADS = 6
NUM_TRANSFORMER_BLOCKS = 8
DROPOUT = 0.1
N_CLASSES = 10
LR = 3e-4

In [7]:
from torch.utils.data import Subset

train_full = datasets.CIFAR10(
    root="../../../data",
    train=True,
    download=False,
    transform=train_transform,
)
val_full = datasets.CIFAR10(
    root="../../../data",
    train=True,
    download=False,
    transform=eval_transform,
)

val_size = 5_000
generator = torch.Generator().manual_seed(42)
permutation = torch.randperm(len(train_full), generator=generator)
val_idx = permutation[:val_size].tolist()
train_idx = permutation[val_size:].tolist()

train_dataset = Subset(train_full, train_idx)
val_dataset = Subset(val_full, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

model = VisionTransformer(
    image_size=IMAGE_SIZE,
    patch_size=PATCH_SIZE,
    in_channels=IN_CHANNELS,
    emb_size=EMB_SIZE,
    num_attention_heads=NUM_ATTENTION_HEADS,
    dropout=DROPOUT,
    num_transformer_blocks=NUM_TRANSFORMER_BLOCKS,
    n_classes=N_CLASSES
).to(device)

Training on: cuda


In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR
)

In [10]:
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 14,272,138


## Small overfitting test

In [ ]:
# Use images without random augmentation
images, labels = next(iter(train_loader))

images = images[:16].to(device)
labels = labels[:16].to(device)

for step in range(300):
    model.train()

    optimizer.zero_grad()

    logits = model(images)
    loss = criterion(logits, labels)

    loss.backward()
    optimizer.step()

    if step % 25 == 0:
        accuracy = (logits.argmax(dim=1) == labels).float().mean()

        print(
            f"Step {step:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Accuracy: {accuracy.item() * 100:.1f}%"
        )

Step   0 | Loss: 2.5061 | Accuracy: 6.2%
Step  25 | Loss: 0.0043 | Accuracy: 100.0%
Step  50 | Loss: 0.0011 | Accuracy: 100.0%


In [ ]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for images, labels in train_loader:
        # STEP 1 - Put them on device
        images = images.to(device)
        labels = labels.to(device)
    
        # STEP 2 - Forward pass
        logits = model(images)
    
        # STEP 3 - Compute loss
        optimizer.zero_grad()
        loss = criterion(logits, labels)
    
        # STEP 4 - Backpropagation
        loss.backward()
    
        # STEP 5 - Update step
        optimizer.step()
    
        # STEP 6 - Metrics
        batch_size = len(labels)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_samples += batch_size
    
    epoch_loss = total_loss / total_samples
    epoch_accuracy = total_correct / total_samples
    
    return epoch_loss, epoch_accuracy

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass only
        logits = model(images)

        # Calculate loss
        loss = criterion(logits, labels)

        # Calculate accuracy
        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy


In [ ]:
import time

num_epochs = 50

for epoch in range(num_epochs):
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    if device.type == "cuda":
        torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - start

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc * 100:.2f}% | "
        f"Time: {epoch_seconds:.1f}s"
    )

Epoch 1/50 | Train Loss: 1.9693 | Train Acc: 26.08% | Val Loss: 1.8556 | Val Acc: 29.74% | Time: 27.4s
Epoch 2/50 | Train Loss: 1.7585 | Train Acc: 33.68% | Val Loss: 1.7483 | Val Acc: 35.26% | Time: 25.4s
Epoch 3/50 | Train Loss: 1.6751 | Train Acc: 37.57% | Val Loss: 1.6403 | Val Acc: 39.06% | Time: 23.4s
Epoch 4/50 | Train Loss: 1.6103 | Train Acc: 40.30% | Val Loss: 1.6188 | Val Acc: 41.56% | Time: 23.5s
Epoch 5/50 | Train Loss: 1.5665 | Train Acc: 42.41% | Val Loss: 1.5849 | Val Acc: 42.28% | Time: 24.7s


In [16]:
checkpoint_dir = project_root / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "vit_cifar10.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "image_size": IMAGE_SIZE,
            "patch_size": PATCH_SIZE,
            "in_channels": IN_CHANNELS,
            "emb_size": EMB_SIZE,
            "num_attention_heads": NUM_ATTENTION_HEADS,
            "num_transformer_blocks": NUM_TRANSFORMER_BLOCKS,
            "dropout": DROPOUT,
            "n_classes": N_CLASSES,
        },
    },
    checkpoint_path,
)
print(f"Saved weights to {checkpoint_path}")

Saved weights to /bigdata/md5_t1fada40cc125fa5312a398ad7cb/experiments/vit-cifar10/checkpoints/vit_cifar10.pt
